In [ ]:
from typing import List, TypedDict
from pydantic import BaseModel
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
import re
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# 1. Document Loading
docs = (
    PyPDFLoader(".documents/book1.pdf").load() +
    PyPDFLoader(".documents/book2.pdf").load() +
    PyPDFLoader(".documents/book3.pdf").load()
)

In [ ]:
len(docs)

In [ ]:
# 2. Chunk
chunks = RecursiveCharacterTextSplitter(chunk_size = 900, chunk_overlap = 150).split_documents(docs)

# 3. Clean text to avoid UnicodeEncodeError (surrogates from PDF extraction)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [ ]:
len(chunks)

In [ ]:
# 4. Index (fresh collection each run)
embeddings = OpenAIEmbeddings(model = "text-embedding-3-large")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
# 5. retriever
retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k': 4})

In [ ]:
# 6. LLM + prompt
llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0)

In [ ]:
UPPER_TH = 0.7
LOWER_TH = 0.3

In [ ]:
class State(TypedDict):
    question: str
    docs: List[Document]

    good_docs: List[Document]
    verdict: str
    reason: str

    strips: List[str] # output of decomposition (sentence strips)
    kept_strips: List[str] # after filtering (kept sentences)
    refined_context: str # recomposed internal knowledge (joined kept_strips)

    answer: str

In [ ]:
def retrieve_node(state):
    q = state["question"]
    return {"docs": retriever.invoke(q)}

In [ ]:
# 3 score-based doc evaluator
class DocEvalScore(BaseModel):
    score: float
    reason: str

doc_eval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "you are a strict retrieval evaluator for RAG.\n"
            "You will be given ONE retrieved chunk and a question.\n"
            "Return a relevance score in [0.0, 0.1].\n"
            "- 1.0: chunk alone is sufficient to answer fully/mostly\n"
            "- 0.0: chunk is irrelevant\n"
            "Be conservative with high scores.\n"
            "Output JSON only."
        ),
        ("human", "Question: {question}\n\nChunk:\n{chunk}")
    ]
)

doc_eval_chain = doc_eval_prompt | llm.with_structured_output(DocEvalScore)

In [ ]:
def eval_each_doc_node(state: State) -> State:
     

    q = state["question"]

    scores: List[float] = []
    reasons: List[str] = []
    good: List[Document] = []

    for d in state["docs"]:
        out = doc_eval_chain.invoke({"question": q, "chunk": d.page_content})
        scores.append(out.score)
        reasons.append(out.reason)

        # for CORRECT case we will refine only docs with score > UPPER_TH
        if out.score > LOWER_TH:
            good.append(d)

    # CORRECT is at least one doc > UPPER_TH
    if any(s > UPPER_TH for s in scores):
        return {
            "good_docs": good,
            "verdict": "CORRECT",
            "reason": f"At least one retrieved chunk scored > {UPPER_TH}."        
        }
    
    # INCORRECT is at least one doc < LOWER_TH
    if len(scores) > 0 and all(s < LOWER_TH for s in scores):
        why = "No chunk was sufficient."
        return {
            "good_docs": [],
            "verdict": "INCORRECT",
            "reason": f"All retrieved chunks scored < {LOWER_TH}. {why}"        
        }
    
    # anything in bewteen => AMBIGUOUS
    why = "Mixed relevance signals"
    return {
        "good_docs": good,
        "verdict": "AMBIGUOUS",
        "reason": f"No chunk scored > {UPPER_TH}, but no all were < {LOWER_TH}. {why}"        
    }

In [ ]:
# sentence elvel decomposer
def decompose_to_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

In [ ]:
# filter (LLM Judge)
class KeepOrDrop(BaseModel):
    keep: bool

In [ ]:
filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a strict relevance filter.\n"
            "return keep=true only if the sentence directly hlps answer the question.\n"
            "Use ONLY the sentence.Output JSON only."
        ),
        ("human", "Question: {question}\n\nsentence:\n{sentence}")
    ]
)

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)

# refining (Decompose -> Filter -> Recompose)
def refine(state: State) -> State:
    q = state["question"]

    # combine retrieved docs into one context string
    context = "\n\n".join(d.page_content for d in state["good_docs"]).strip()

    # 1. DECOMPOSITION: context -> sentence strips
    strips = decompose_to_sentences(context)

    # 2. FILTER: keep only relevance strips
    kept: List[str] = []

    for s in strips:
        if filter_chain.invoke({"question": q, "sentence": s}).keep:
            kept.append(s)

    # 3. RECOMPOSE: glue kept strips back together (internal knowledge)
    refined_context = "\n".join(kept).strip()

    return {
        "strips": strips,
        "kept_strips": kept,
        "refined_context": refined_context
    }

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "You are a helpful Ml tutor. Answer ONLY using the provided refined bullets.\n"
         "If the bullets are empty or insufficient, say: 'I don't know based on the provided books.'")
         ,
        ("human", "Question: {question}\n\nRefined Context:\n{refined_context}")
    ]
)

def generate(state):
    out = (answer_prompt | llm).invoke({"question": state["question"], "refined_context": state['refined_context']})
    return {"answer": out.content}

In [ ]:
def fail_node(state: State) -> State:
    return {"answer": f"FAIL: {state['reason']}"}

def ambiguous_node(state: State) -> State:
    return {"answer": f"Ambiguous: {state['reason']}"}

def route_after_eval(state: State) -> str:
    if state['verdict'] == 'CORRECT':
        return "refine"
    elif state['verdict'] == "INCORRECT":
        return "web_search"
    else:
        return "ambiguous"

In [ ]:
g = StateGraph(State)
g.add_node("retrieve", retrieve_node)
g.add_node("eval_each_doc", eval_each_doc_node)
g.add_node("refine", refine)
g.add_node("generate", generate)
g.add_node("fail", fail_node)
g.add_node("ambiguous", ambiguous_node)

g.add_edge(START, "retrieve")
g.add_edge("retrieve", "eval_each_doc")

g.add_conditional_edges(
    "eval_each_doc",
    route_after_eval,
    {"refine": "refine", "web_search": "fail", "ambiguous": "ambiguous"}
)
g.add_edge("refine", "generate")
g.add_edge("generate", END)
g.add_edge("fail", END)

app = g.compile()
app

In [ ]:
res = app.invoke({
    "question": "What are attention mechanism and why they are important in current nodes",
     "docs": [], 
     "good_docs": [], 
     "verdict": "", 
     "reason": "", 
     "strips": [],
     "keep_strips": [],
     "refined_context": "",
     "answer": ""
    })

print("VERDICT:\n", res["verdict"])
print("REASON:\n", res["reason"])
print("\nOUTPUT:\n", res["answer"])

In [ ]:
print(res['docs'][0].page_content)
print('*'*100)
print(res['docs'][1].page_content)
print('*'*100)
print(res['docs'][2].page_content)
print('*'*100)
print(res['docs'][3].page_content)
print('*'*100)

In [ ]:
print(res['kept_strings'])

In [ ]:
## Try these queries
# AI news from latest week
# Bias-variance trade-off